# Subsystems

Companion notebook for Tutorial 10. Use `surge.subsystem.Subsystem` to
create lightweight, repeatable bus-set filters over a network.

## Basic Filters

Filter by voltage level, area, or explicit bus list. All criteria
intersect — a bus must satisfy every filter to be included.

In [ ]:
import surge

net = surge.case118()

ehv = surge.subsystem.Subsystem(net, name="ehv", kv_min=345.0)
area_one = surge.subsystem.Subsystem(net, name="area-1", areas=[1])
load_pocket = surge.subsystem.Subsystem(net, name="load-pocket", buses=[8, 30, 38])

print(f"EHV buses ({len(ehv.bus_numbers)}): {ehv.bus_numbers[:10]}")
print(f"Area 1 buses ({len(area_one.bus_numbers)}): {area_one.bus_numbers[:10]}")
print(f"Load pocket buses: {load_pocket.bus_numbers}")

## Elements and Totals

Each subsystem exposes generation, load, branch, and tie-line summaries.

In [ ]:
for sub in [ehv, area_one, load_pocket]:
    print(f"\n--- {sub.name} ---")
    print(f"  Buses:      {len(sub.bus_numbers)}")
    print(f"  Branches:   {len(sub.branches)}")
    print(f"  Tie lines:  {len(sub.tie_branches)}")
    print(f"  Generators: {len(sub.generators)}")
    print(f"  Load (MW):  {sub.total_load_mw:.1f}")
    print(f"  Gen  (MW):  {sub.total_generation_mw:.1f}")

## Intersecting Filters

Combine area, voltage, and bus-type filters to narrow down to a specific
subset.

In [ ]:
sub = surge.subsystem.Subsystem(
    net,
    name="area-1-ehv-load",
    areas=[1],
    kv_min=230.0,
    bus_type="PQ",
)

print(f"Buses matching area=1, kV>=230, type=PQ: {sub.bus_numbers}")
print(f"Total load: {sub.total_load_mw:.1f} MW")

## Live Values, Fixed Bus Set

The bus set is fixed at creation time, but aggregates reflect the current
network state. Changing dispatch or load updates the totals automatically.

In [ ]:
print(f"Area 1 load before: {area_one.total_load_mw:.1f} MW")

net.scale_loads(1.10)

print(f"Area 1 load after 10% scale: {area_one.total_load_mw:.1f} MW")